In [ ]:
# !pip install -q transformers

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import random
import re
from collections import Counter
from transformers import pipeline

# Set device to GPU if available in Colab, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")

Using device: cuda



In [ ]:
# ==========================================
# STEP 2: Simulate Real Data
# ==========================================
print("Generating 1,000 synthetic movie reviews...")
positive_keywords = ["great", "awesome", "fantastic", "loved", "excellent", "amazing", "beautiful", "brilliant", "perfect", "enjoyed"]
negative_keywords = ["terrible", "awful", "bad", "hated", "worst", "boring", "dull", "poor", "garbage", "waste"]
neutral_fillers   = ["the movie was", "i thought it was", "the acting was", "overall it was", "the plot was", "the director made it"]

data = []
for _ in range(500):
    # Generate Positive Reviews (Label 1)
    pos_review = f"{random.choice(neutral_fillers)} {random.choice(positive_keywords)} and {random.choice(positive_keywords)}"
    data.append({"text": pos_review, "label": 1})

    # Generate Negative Reviews (Label 0)
    neg_review = f"{random.choice(neutral_fillers)} {random.choice(negative_keywords)} and {random.choice(negative_keywords)}"
    data.append({"text": neg_review, "label": 0})

# Shuffle the dataset
random.shuffle(data)
df = pd.DataFrame(data)

Generating 1,000 synthetic movie reviews...


In [ ]:
df.head()

,text,label
0,i thought it was amazing and loved,1
1,i thought it was waste and awful,0
2,the movie was garbage and worst,0
3,overall it was beautiful and amazing,1
4,i thought it was amazing and awesome,1


In [ ]:
df.shape

(1000, 2)

In [ ]:
df['label'].value_counts()

,count
label,
1,500
0,500


In [ ]:
df.columns

Index(['text', 'label'], dtype='object')

In [ ]:
# ==========================================
# STEP 3: Data Preprocessing & Tokenization
# ==========================================
print("Preprocessing and Tokenizing data...")

# Clean Text (lowercase and remove punctuation)
def clean_text(text):
    text = text.lower()
    return re.sub(r'[^a-z\s]', '', text)

df['clean_text'] = df['text'].apply(clean_text)

# Build a Vocabulary
all_words = ' '.join(df['clean_text']).split()
vocab_counts = Counter(all_words)
# Map words to IDs (start from 2 to reserve 0 for PAD and 1 for UNK)
vocab = {word: i+2 for i, (word, count) in enumerate(vocab_counts.items())}
vocab['<PAD>'] = 0
vocab['<UNK>'] = 1


# Convert Sentence to Sentence IDs
def tokenize(text, max_length=10):
    tokens = [vocab.get(word, vocab['<UNK>']) for word in text.split()]
    # Pad sequences so they are all the same length (required for tensors)
    if len(tokens) < max_length:
        tokens.extend([vocab['<PAD>']] * (max_length - len(tokens)))
    return tokens[:max_length]

df['tokens'] = df['clean_text'].apply(tokenize)

# Prepare pytorch sensors
X = torch.tensor(df['tokens'].tolist(), dtype = torch.long).to(device)
y = torch.tensor(df['label'].tolist(), dtype=torch.float32).unsqueeze(1).to(device)

Preprocessing and Tokenizing data...


In [ ]:
# ==========================================
# STEP 4: Build the Custom SLM Architecture
# ==========================================
class CustomSLM(nn.Module) :
  def __init__(self, vocab_size,embed_dim,hidden_dim, output_dim):
    super(CustomSLM,self).__init__()
    self.embedding = nn.Embedding(vocab_size,embed_dim,padding_idx=0)
    self.fc1  = nn.Linear(embed_dim, hidden_dim)
    self.relu = nn.ReLU()
    self.fc2  = nn.Linear(hidden_dim,output_dim)
    self.sigmoid = nn.Sigmoid()

  def forward(self, text):
    embedded = self.embedding(text)
    # average
    pooled = embedded.mean(dim=1)
    hidden = self.relu(self.fc1(pooled))
    output = self.sigmoid(self.fc2(hidden))
    return output

In [ ]:
VOCAB_SIZE = len(vocab)
model = CustomSLM(vocab_size = VOCAB_SIZE, embed_dim = 16, hidden_dim = 32,
                  output_dim=1).to(device)

In [ ]:
# ==========================================
# STEP 5: Train the Custom Model
# ==========================================
print("\nTraining the Custom SLM from scratch...")
criterion = nn.BCELoss() # Binary Cross Entropy for 0/1 classification
optimizer = optim.Adam(model.parameters(), lr=0.01)
epochs = 15

for epoch in range(epochs):
  model.train()
  optimizer.zero_grad()

  predictions = model(X)
  loss = criterion(predictions, y)
  loss.backward()
  optimizer.step()

  if (epoch+1) %3 == 0 :
    # Accuracy (basic)
    predicted_classes = (predictions > 0.5).float()
    accuracy = (predicted_classes == y).sum().item()/len(y)
    print(f"Epoch: {epoch+1:02d}/{epochs} | Loss: {loss.item():.4f} | Accuracy: {accuracy*100:.1f}%")

print("Training Completed! \n")


Training the Custom SLM from scratch...
Epoch: 03/15 | Loss: 0.6607 | Accuracy: 68.8%
Epoch: 06/15 | Loss: 0.6201 | Accuracy: 85.3%
Epoch: 09/15 | Loss: 0.5598 | Accuracy: 93.1%
Epoch: 12/15 | Loss: 0.4777 | Accuracy: 97.1%
Epoch: 15/15 | Loss: 0.3773 | Accuracy: 98.7%
Training Completed! 



In [ ]:
# ==========================================
# STEP 6: Compare with Pre-Trained DistilBERT
# ==========================================
print("-" * 50)
print("Evaluating Sample Sentences")
print("-" * 50)
# Sample sentences to test both models
test_sentences = [
    "the movie was absolutely fantastic and brilliant",
    "i thought it was terrible and a waste",
    "it was so good that it cured my insomnia",
    "What a fantastic way to ruin my day"
]

# 1. Test Custom Model
print(">>> Custom SLM Predictions:")
model.eval()
with torch.no_grad():
    for sentence in test_sentences:
      clean_s = clean_text(sentence)
      tokens = tokenize(clean_s)
      tensor_input = torch.tensor([tokens], dtype=torch.long).to(device)
      prediction = model(tensor_input).item()
      sentiment = "Positive" if prediction > 0.5 else "Negative"
      print(f"Text: '{sentence}' \n-> Sentiment: {sentiment} (Score: {prediction:.4f})\n")

--------------------------------------------------
Evaluating Sample Sentences
--------------------------------------------------
>>> Custom SLM Predictions:
Text: 'the movie was absolutely fantastic and brilliant' 
-> Sentiment: Positive (Score: 0.7314)

Text: 'i thought it was terrible and a waste' 
-> Sentiment: Negative (Score: 0.4414)

Text: 'it was so good that it cured my insomnia' 
-> Sentiment: Positive (Score: 0.9488)

Text: 'What a fantastic way to ruin my day' 
-> Sentiment: Positive (Score: 0.9475)



In [ ]:
# 2. Test DistilBERT (Pre-Trained)
print(">>> DistilBERT Predictions (Downloading model if first run...):")
pipe_device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device = pipe_device)

>>> DistilBERT Predictions (Downloading model if first run...):


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

In [ ]:
results = classifier(test_sentences)
for sentence, result in zip(test_sentences, results):
    print(f"Text: '{sentence}' \n-> Sentiment: {result['label']} (Confidence: {result['score']:.4f})\n")

Text: 'the movie was absolutely fantastic and brilliant' 
-> Sentiment: POSITIVE (Confidence: 0.9999)

Text: 'i thought it was terrible and a waste' 
-> Sentiment: NEGATIVE (Confidence: 0.9998)

Text: 'it was so good that it cured my insomnia' 
-> Sentiment: POSITIVE (Confidence: 0.9997)

Text: 'What a fantastic way to ruin my day' 
-> Sentiment: NEGATIVE (Confidence: 0.9938)

